[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/01-python/11_oop.ipynb)

*Python from Scratch · Notebook 11 of 12*

# Object-Oriented Programming

Model things as objects: classes, attributes, methods, encapsulation, inheritance, polymorphism and special methods.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A class is a cookie cutter and objects are the cookies. Each cookie has its own data but shares the same behaviour.

**What interviewers really test here:** Inheritance vs composition, dunder methods, properties and class vs instance attributes.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Classes and objects *(Basics)*
- [ ] 2. Methods that change an object's state *(Basics)*
- [ ] 3. Class attributes, @classmethod and @staticmethod *(Core)*
- [ ] 4. Encapsulation and properties *(Core)*
- [ ] 5. Inheritance and polymorphism *(Core)*
- [ ] 6. Special (dunder) methods *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline

PLAIN = False

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Definition

**Object-oriented programming (OOP)** is a programming paradigm that organises software around **objects** — bundles of data (**attributes**, state) and behaviour (**methods**) — rather than around functions and logic alone.




- A **class** is a user-defined type: the blueprint describing what attributes and methods its objects have.
- An **object** (instance) is a concrete value created from a class, with its own attribute values.
- In Python *everything* is an object — ints, functions and classes themselves.

### The four pillars

| Pillar | Meaning | In Python |
| --- | --- | --- |
| **Encapsulation** | bundle data with the methods that operate on it; restrict direct access | methods, `_internal` naming, `__name_mangling`, `@property` |
| **Abstraction** | expose what an object does, hide how | clear public methods; abstract base classes (`abc.ABC`, `@abstractmethod`) |
| **Inheritance** | a class reuses and extends another (IS-A) | `class Dog(Animal)`, `super()`, multiple inheritance with MRO |
| **Polymorphism** | one interface, many implementations | method overriding, duck typing, operator overloading |

### Class anatomy

| Element | Description |
| --- | --- |
| `__init__(self, …)` | initialiser, runs after the object is created (the constructor proper is `__new__`) |
| `self` | reference to the current instance; first parameter of instance methods (a convention, not a keyword) |
| Instance attribute | `self.x = …` — stored per object (in `obj.__dict__`) |
| Class attribute | assigned in the class body — shared by all instances |
| Instance method | receives `self` |
| `@classmethod` | receives the class (`cls`); used for factory constructors |
| `@staticmethod` | receives nothing implicit; a namespaced helper |
| `@property` | method accessed like an attribute; optional setter with `@x.setter` |



Attribute lookup on `obj.name`: the instance's `__dict__` first, then the class, then the parent classes in **MRO** (method resolution order, see `Class.__mro__`).

### Inheritance in detail

| Type | Shape |
| --- | --- |
| Single | B(A) |
| Multilevel | C(B), B(A) |
| Hierarchical | B(A), C(A) |
| Multiple | C(A, B) |
| Hybrid | a combination (e.g. diamond: D(B, C), B(A), C(A)) |




- **Overriding**: a subclass defines a method with the same name; `super().method()` calls the parent's version.
- Python resolves multiple inheritance with the **C3 linearisation** (MRO), so each class appears once in a predictable order.
- `isinstance(obj, Class)` and `issubclass(B, A)` test relationships.
- **Composition over inheritance**: prefer HAS-A relationships unless IS-A is genuinely true.

### Polymorphism and duck typing

Python is **duck-typed**: "if it walks like a duck and quacks like a duck, it's a duck". Code works with any object that has the needed methods, regardless of its class. That's why `len()` works on strings, lists and your own classes with `__len__`.



Python does not support method **overloading** by signature (defining the same method twice with different parameters). Use default values, `*args`, or `functools.singledispatch` instead.



#### Common special methods



`__init__ __repr__ __str__ __eq__ __lt__ __hash__ __len__ __contains__ __getitem__ __setitem__ __iter__ __next__ __add__ __sub__ __mul__ __call__ __enter__ __exit__ __bool__`



Rule: if you define `__eq__`, define `__hash__` too (or the objects become unhashable).

### Dataclasses and when to use OOP

`@dataclass` (Python 3.7+) generates `__init__`, `__repr__` and `__eq__` from annotated fields; options add ordering (`order=True`), immutability (`frozen=True`) and defaults.



```python
from dataclasses import dataclass, field
@dataclass
class Item:
    name: str
    price: float = 0.0
    tags: list = field(default_factory=list)
```



Use classes when data and the rules for changing it belong together, when you have many similar things, or when you need several interchangeable implementations. For a simple bag of data, a dict, tuple or dataclass is enough.

---
# Part 2: Lessons, step by step

## Lesson 1: Classes and objects

*Level: Basics*

So far data (variables) and actions (functions) lived separately. **Object-oriented programming** bundles them together. A **class** is a blueprint; an **object** (instance) is one thing built from it.



```python
class Dog:                         # the blueprint
    def __init__(self, name):      # runs when a Dog is created
        self.name = name           # an attribute stored on this object
    def bark(self):                # a method: a function that belongs to the class
        return self.name + " says woof"

rex = Dog("Rex")                   # create an object
rex.bark()                         # "Rex says woof"
```



`self` is the object the method was called on. `rex.bark()` is really `Dog.bark(rex)`. In the animation, the box for `self` shows the object's attributes as they're set.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def class_demo():
    class Dog:
        def __init__(self, name):
            self.name = name
            self.tricks = []

        def learn(self, trick):
            self.tricks.append(trick)

    rex = Dog("Rex")
    rex.learn("sit")
    rex.learn("roll")
    fido = Dog("Fido")
    print(rex.name, rex.tricks, fido.tricks)


class_demo()

Rex ['sit', 'roll'] []


<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Dog() returns None
2. \_\_init\_\_() returns None
3. learn() returns None
4. learn() returns None
5. \_\_init\_\_() returns None
6. class\_demo() returns None

</details>


**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q3](#q3). Try one now, or carry on and do them in Part 3.

## Lesson 2: Methods that change an object's state

*Level: Basics*

An object remembers its attributes between method calls, so methods can update them. That's the main reason to use a class: the data and the rules for changing it stay together.



```python
class BankAccount:
    def __init__(self, balance=0):
        self.balance = balance
    def deposit(self, amount):
        self.balance += amount
    def withdraw(self, amount):
        if amount > self.balance:
            return False
        self.balance -= amount
        return True
```



Each object has its **own** attributes: two accounts don't share a balance.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def state_demo():
    class BankAccount:
        def __init__(self, balance=0):
            self.balance = balance

        def deposit(self, amount):
            self.balance += amount

        def withdraw(self, amount):
            if amount > self.balance:
                return False
            self.balance -= amount
            return True

    acc = BankAccount(100)
    acc.deposit(50)
    ok = acc.withdraw(500)
    print(acc.balance, ok)


state_demo()

150 False


<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. BankAccount() returns None
2. \_\_init\_\_() returns None
3. deposit() returns None
4. withdraw() returns False
5. state\_demo() returns None

</details>


**✏️ Your turn:** you now have what you need for [Q2](#q2), [Q4](#q4), [Q11](#q11), [Q12](#q12). Try one now, or carry on and do them in Part 3.

## Lesson 3: Class attributes, @classmethod and @staticmethod

*Level: Core*

- **Instance attributes** (`self.x`) belong to one object.
- **Class attributes** (written in the class body) are shared by all objects — e.g. a counter of instances.
- `@classmethod` receives the class (`cls`) instead of an object — used for alternative constructors like `Date.from_string("25-12-2024")`.
- `@staticmethod` receives neither — a plain helper that lives in the class's namespace.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [4]:
def class_attr_demo():
    class Robot:
        count = 0

        def __init__(self, name):
            self.name = name
            Robot.count += 1

        @classmethod
        def from_code(cls, code):
            return cls("R-" + code)

    a = Robot("Ava")
    b = Robot.from_code("42")
    print(Robot.count, b.name)


class_attr_demo()

2 R-42


<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Robot() returns None
2. \_\_init\_\_() returns None
3. \_\_init\_\_() returns None
4. from\_code() returns &lt;lessons\_11\_oop.class\_attr\_demo.&lt;loca...
5. class\_attr\_demo() returns None

</details>


**✏️ Your turn:** you now have what you need for [Q17](#q17), [Q19](#q19). Try one now, or carry on and do them in Part 3.

## Lesson 4: Encapsulation and properties

*Level: Core*

**Encapsulation** means keeping an object's data valid by changing it only through its methods. Python has no truly private attributes; by convention a leading underscore (`self._age`) means "internal, don't touch from outside".



A **property** lets you compute or validate a value while still reading it like an attribute:



```python
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius
    @property
    def fahrenheit(self):              # read as t.fahrenheit (no brackets)
        return self.celsius * 9 / 5 + 32
```

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [5]:
def property_demo():
    class Temperature:
        def __init__(self, celsius):
            self.celsius = celsius

        @property
        def fahrenheit(self):
            return self.celsius * 9 / 5 + 32

    t = Temperature(25)
    before = t.fahrenheit
    t.celsius = 100
    after = t.fahrenheit
    print(before, after)


property_demo()

77.0 212.0


<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Temperature() returns None
2. \_\_init\_\_() returns None
3. fahrenheit() returns 77.0
4. fahrenheit() returns 212.0
5. property\_demo() returns None

</details>


**✏️ Your turn:** you now have what you need for [Q9](#q9), [Q13](#q13). Try one now, or carry on and do them in Part 3.

## Lesson 5: Inheritance and polymorphism

*Level: Core*

**Inheritance**: a class can extend another, reusing its attributes and methods. The new class can **override** methods and call the parent's version with `super()`.



```python
class Animal:
    def __init__(self, name): self.name = name
    def speak(self): return "..."
class Dog(Animal):                  # Dog IS-A Animal
    def speak(self): return "Woof"
```



**Polymorphism**: code that calls `animal.speak()` works for any animal, and each object answers in its own way — no if/elif on the type needed.



**Composition** ("HAS-A") is the alternative: a Car *has* an Engine object. Prefer composition when the relationship isn't truly "is a kind of".

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [6]:
def inheritance_demo():
    class Animal:
        def __init__(self, name):
            self.name = name

        def speak(self):
            return "..."

        def intro(self):
            return f"{self.name}: {self.speak()}"

    class Dog(Animal):
        def speak(self):
            return "Woof"

    class Cat(Animal):
        def speak(self):
            return "Meow"

    for pet in [Dog("Rex"), Cat("Tom"), Animal("Fish")]:
        print(pet.intro())


inheritance_demo()

Rex: Woof
Tom: Meow
Fish: ...


<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Animal() returns None
2. Dog() returns None
3. Cat() returns None
4. \_\_init\_\_() returns None
5. \_\_init\_\_() returns None
6. \_\_init\_\_() returns None
7. speak() returns 'Woof'
8. intro() returns 'Rex: Woof'
9. speak() returns 'Meow'
10. intro() returns 'Tom: Meow'
11. speak() returns '...'
12. intro() returns 'Fish: ...'
13. inheritance\_demo() returns None

</details>


**✏️ Your turn:** you now have what you need for [Q7](#q7), [Q16](#q16), [Q20](#q20). Try one now, or carry on and do them in Part 3.

## Lesson 6: Special (dunder) methods

*Level: Advanced*

Methods with double underscores let your objects work with Python's built-in syntax:



| Method | Makes this work |
| --- | --- |
| `__init__` | `Obj(args)` — set up a new object |
| `__str__` / `__repr__` | `print(obj)` / how it looks in the shell and in lists |
| `__eq__`, `__lt__` | `a == b`, `a < b`, and therefore `sorted()` |
| `__add__`, `__mul__` … | `a + b`, `a * b` (operator overloading) |
| `__len__`, `__contains__` | `len(obj)`, `x in obj` |
| `__getitem__`, `__iter__` | `obj[i]`, `for x in obj` |



`@dataclass` writes `__init__`, `__repr__` and `__eq__` for you from type-annotated fields.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [7]:
def dunder_demo():
    class Vector:
        def __init__(self, x, y):
            self.x, self.y = x, y

        def __add__(self, other):
            return Vector(self.x + other.x, self.y + other.y)

        def __eq__(self, other):
            return (self.x, self.y) == (other.x, other.y)

        def __repr__(self):
            return f"Vector({self.x}, {self.y})"

    v = Vector(1, 2) + Vector(3, 4)
    print(v, v == Vector(4, 6))


dunder_demo()

Vector(4, 6) True


<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Vector() returns None
2. \_\_init\_\_() returns None
3. \_\_init\_\_() returns None
4. \_\_init\_\_() returns None
5. \_\_add\_\_() returns Vector(4, 6)
6. \_\_init\_\_() returns None
7. \_\_eq\_\_() returns True
8. \_\_repr\_\_() returns 'Vector(4, 6)'
9. dunder\_demo() returns None

</details>


**✏️ Your turn:** you now have what you need for [Q15](#q15), [Q6](#q6), [Q14](#q14), [Q18](#q18), [Q8](#q8). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [8]:
from dataclasses import dataclass


<a id="q1"></a>
### Q1. Rectangle Class

🟢 Warm-up · *Attributes and methods*

Create a `Rectangle(width, height)` class with methods `area()` and `perimeter()`.

**Examples**

- `Rectangle(3, 4) → area() → perimeter()` → `[12, 14]`
- `Rectangle(5, 5) → area()` → `[25]`

In [ ]:
class Rectangle:
    """
    Create a Rectangle(width, height) class with methods area() and perimeter().
    """

    def __init__(self, width: float, height: float) -> None:
        pass

    def area(self):
        pass

    def perimeter(self):
        pass


# ▶ Run this cell to test your answer
check(Rectangle, [
    (((3, 4), [['area', ()], ['perimeter', ()]]), [12, 14]),
    (((5, 5), [['area', ()]]), [25]),
], design=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Store the values and compute everything step by step inside each method.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A `@dataclass` writes `__init__` for you; methods are one line each.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Store the values and compute everything step by step inside each method.

```python
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        result = self.width * self.height
        return result

    def perimeter(self):
        result = 0
        result = result + self.width + self.width
        result = result + self.height + self.height
        return result
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A `@dataclass` writes `__init__` for you; methods are one line each.

```python
@dataclass
class Rectangle:
    width: float
    height: float

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Store the values and compute everything step by step inside each method. That costs O(1) time and O(1) space."

"I can do better: A `@dataclass` writes `__init__` for you; methods are one line each. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/oop/rectangle-class)

<a id="q2"></a>
### Q2. Bank Account

🟢 Warm-up · *Methods that change state*

`BankAccount(balance)` with `deposit(x)`, `withdraw(x)` (returns False and changes nothing if funds are insufficient, else True) and `get_balance()`.

**Examples**

- `BankAccount(100) → deposit(50) → withdraw(30) → withdraw(500) → get_balance()` → `[None, True, False, 120]`

In [ ]:
class BankAccount:
    """
    BankAccount(balance) with deposit(x), withdraw(x) (returns False and changes
    nothing if funds are insufficient, else True) and get_balance().
    """

    def __init__(self, balance):
        pass

    def deposit(self, x):
        pass

    def withdraw(self, x):
        pass

    def get_balance(self):
        pass


# ▶ Run this cell to test your answer
check(BankAccount, [
    (((100,), [['deposit', (50,)], ['withdraw', (30,)], ['withdraw', (500,)], ['get_balance', ()]]), [None, True, False, 120]),
], design=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Keep a list of every transaction and recompute the balance from it each time.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Store the current balance and update it directly.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Keep a list of every transaction and recompute the balance from it each time.

```python
class BankAccount:
    def __init__(self, balance):
        self.transactions = [balance]

    def deposit(self, x):
        self.transactions.append(x)

    def withdraw(self, x):
        if sum(self.transactions) < x:
            return False
        self.transactions.append(-x)
        return True

    def get_balance(self):
        return sum(self.transactions)
```

**Cost:** O(t) per call time, O(t) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Store the current balance and update it directly.

```python
class BankAccount:
    def __init__(self, balance):
        self.balance = balance

    def deposit(self, x):
        self.balance += x

    def withdraw(self, x):
        if x > self.balance:
            return False
        self.balance -= x
        return True

    def get_balance(self):
        return self.balance
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Keep a list of every transaction and recompute the balance from it each time. That costs O(t) per call time and O(t) space."

"I can do better: Store the current balance and update it directly. That's O(1) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `BankAccount(100) → deposit(50) → withdraw(30) → withdraw(500) → get_balance()`

1. \_\_init\_\_() returns None
2. deposit() returns None
3. withdraw() returns True
4. withdraw() returns False
5. get\_balance() returns 120

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/oop/bank-account)

<a id="q3"></a>
### Q3. Click Counter

🟢 Warm-up · *Instance state*

`Counter()` with `click()` (adds 1 and returns the new count), `reset()` and `value()`.

**Examples**

- `Counter() → click() → click() → value() → reset() → click()` → `[1, 2, 2, None, 1]`

In [ ]:
class Counter:
    """
    Counter() with click() (adds 1 and returns the new count), reset() and
    value().
    """

    def __init__(self):
        pass

    def click(self):
        pass

    def value(self):
        pass

    def reset(self):
        pass


# ▶ Run this cell to test your answer
check(Counter, [
    (((), [['click', ()], ['click', ()], ['value', ()], ['reset', ()], ['click', ()]]), [1, 2, 2, None, 1]),
], design=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Store every click in a list; the count is its length.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Store just the number.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Store every click in a list; the count is its length.

```python
class Counter:
    def __init__(self):
        self.clicks = []

    def click(self):
        self.clicks.append(1)
        return len(self.clicks)

    def reset(self):
        self.clicks = []

    def value(self):
        return len(self.clicks)
```

**Cost:** O(1) time, O(clicks) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Store just the number.

```python
class Counter:
    def __init__(self):
        self.count = 0

    def click(self):
        self.count += 1
        return self.count

    def reset(self):
        self.count = 0

    def value(self):
        return self.count
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Store every click in a list; the count is its length. That costs O(1) time and O(clicks) space."

"I can do better: Store just the number. That's O(1) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `Counter() → click() → click() → value() → reset() → click()`

1. \_\_init\_\_() returns None
2. click() returns 1
3. click() returns 2
4. value() returns 2
5. reset() returns None
6. click() returns 1

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/oop/click-counter)

<a id="q4"></a>
### Q4. Student Average

🟢 Warm-up · *Lists inside objects*

`Student(name)` with `add_mark(m)` and `average()` (rounded to 2 decimals; 0 if no marks).

**Examples**

- `Student('Ana') → add_mark(80) → add_mark(91) → average()` → `[None, None, 85.5]`
- `Student('Ben') → average()` → `[0]`

In [ ]:
class Student:
    """
    Student(name) with add_mark(m) and average() (rounded to 2 decimals; 0 if no
    marks).
    """

    def __init__(self, name):
        pass

    def add_mark(self, m):
        pass

    def average(self):
        pass


# ▶ Run this cell to test your answer
check(Student, [
    ((('Ana',), [['add_mark', (80,)], ['add_mark', (91,)], ['average', ()]]), [None, None, 85.5]),
    ((('Ben',), [['average', ()]]), [0]),
], design=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Keep a running total and a count, and divide with an if-check.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Keep the marks list; `average` uses sum/len with a guard.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Keep a running total and a count, and divide with an if-check.

```python
class Student:
    def __init__(self, name):
        self.name = name
        self.total = 0
        self.count = 0

    def add_mark(self, m):
        self.total = self.total + m
        self.count = self.count + 1

    def average(self):
        if self.count == 0:
            return 0
        return round(self.total / self.count, 2)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Keep the marks list; `average` uses sum/len with a guard.

```python
class Student:
    def __init__(self, name):
        self.name = name
        self.marks = []

    def add_mark(self, m):
        self.marks.append(m)

    def average(self):
        return round(sum(self.marks) / len(self.marks), 2) if self.marks else 0
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Keep a running total and a count, and divide with an if-check. That costs O(1) time and O(1) space."

"I can do better: Keep the marks list; `average` uses sum/len with a guard. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `Student('Ana') → add_mark(80) → add_mark(91) → average()`

1. \_\_init\_\_() returns None
2. add\_mark() returns None
3. add\_mark() returns None
4. average() returns 85.5

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/oop/student-average)

<a id="q5"></a>
### Q5. Simple Stack Class

🟢 Warm-up · *Wrapping a list*

`Stack()` with `push(x)`, `pop()` (None if empty), `peek()` and `is_empty()`.

**Examples**

- `Stack() → push(1) → push(2) → peek() → pop() → pop() → pop() → is_empty()` → `[None, None, 2, 2, 1, None, True]`

In [ ]:
class Stack:
    """
    Stack() with push(x), pop() (None if empty), peek() and is_empty().
    """

    def __init__(self):
        pass

    def push(self, x):
        pass

    def peek(self):
        pass

    def pop(self):
        pass

    def is_empty(self):
        pass


# ▶ Run this cell to test your answer
check(Stack, [
    (((), [['push', (1,)], ['push', (2,)], ['peek', ()], ['pop', ()], ['pop', ()], ['pop', ()], ['is_empty', ()]]), [None, None, 2, 2, 1, None, True]),
], design=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Track the size in a separate counter and insert at the front of the list.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Use the end of a list: append and pop are O(1).

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Track the size in a separate counter and insert at the front of the list.

```python
class Stack:
    def __init__(self):
        self.items = []
        self.size = 0

    def push(self, x):
        self.items.insert(0, x)
        self.size += 1

    def pop(self):
        if self.size == 0:
            return None
        self.size -= 1
        return self.items.pop(0)

    def peek(self):
        return self.items[0] if self.size else None

    def is_empty(self):
        return self.size == 0
```

**Cost:** O(n) push time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Use the end of a list: append and pop are O(1).

```python
class Stack:
    def __init__(self):
        self.items = []

    def push(self, x):
        self.items.append(x)

    def pop(self):
        return self.items.pop() if self.items else None

    def peek(self):
        return self.items[-1] if self.items else None

    def is_empty(self):
        return not self.items
```

**Cost:** O(1) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Track the size in a separate counter and insert at the front of the list. That costs O(n) push time and O(n) space."

"I can do better: Use the end of a list: append and pop are O(1). That's O(1) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `Stack() → push(1) → push(2) → peek() → pop() → pop() → pop() → is_empty()`

1. \_\_init\_\_() returns None
2. push() returns None
3. push() returns None
4. peek() returns 2
5. pop() returns 2
6. pop() returns 1
7. pop() returns None
8. is\_empty() returns True

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/oop/simple-stack-class)

<a id="q6"></a>
### Q6. Readable Object Text

🟢 Warm-up · *\_\_str\_\_ / \_\_repr\_\_*

Return the text of a Book object as `'TITLE by AUTHOR'`.

**Examples**

- `title = 'Dune', author = 'Herbert'` → `'Dune by Herbert'`

In [ ]:
def book_text(title, author):
    """
    Return the text of a Book object as 'TITLE by AUTHOR'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(book_text, [
    (('Dune', 'Herbert'), 'Dune by Herbert'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A `describe()` method you have to remember to call.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Define `__str__` so `str(book)` and `print(book)` just work.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** A `describe()` method you have to remember to call.

```python
def book_text(title, author):
    class Book:
        def __init__(self, title, author):
            self.title = title
            self.author = author

        def describe(self):
            return self.title + " by " + self.author

    return Book(title, author).describe()
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Define `__str__` so `str(book)` and `print(book)` just work.

```python
def book_text(title, author):
    class Book:
        def __init__(self, title, author):
            self.title, self.author = title, author

        def __str__(self):
            return f"{self.title} by {self.author}"

    return str(Book(title, author))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A `describe()` method you have to remember to call. That costs O(1) time and O(1) space."

"I can do better: Define `__str__` so `str(book)` and `print(book)` just work. That's O(1) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `title = 'Dune', author = 'Herbert'`

1. Book() returns None
2. \_\_init\_\_() returns None
3. \_\_str\_\_() returns 'Dune by Herbert'
4. book\_text() returns 'Dune by Herbert'

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/oop/readable-object-text)

<a id="q7"></a>
### Q7. Animal Sounds

🟢 Warm-up · *Method overriding*

Given animal names ('dog', 'cat', 'cow'), return the list of their sounds ('Woof', 'Meow', 'Moo').

**Examples**

- `names = ['dog', 'cat', 'cow']` → `['Woof', 'Meow', 'Moo']`
- `names = []` → `[]`

In [ ]:
def sounds(names):
    """
    Given animal names ('dog', 'cat', 'cow'), return the list of their sounds
    ('Woof', 'Meow', 'Moo').
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sounds, [
    ((['dog', 'cat', 'cow'],), ['Woof', 'Meow', 'Moo']),
    (([],), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

An if/elif chain on the name.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Each subclass overrides `speak()`; the caller doesn't care which animal it has.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** An if/elif chain on the name.

```python
def sounds(names):
    out = []
    for n in names:
        if n == "dog":
            out.append("Woof")
        elif n == "cat":
            out.append("Meow")
        elif n == "cow":
            out.append("Moo")
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Each subclass overrides `speak()`; the caller doesn't care which animal it has.

```python
def sounds(names):
    class Animal:
        sound = "..."

        def speak(self):
            return self.sound

    class Dog(Animal):
        sound = "Woof"

    class Cat(Animal):
        sound = "Meow"

    class Cow(Animal):
        sound = "Moo"

    kinds = {"dog": Dog, "cat": Cat, "cow": Cow}
    return [kinds[n]().speak() for n in names]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: An if/elif chain on the name. That costs O(n) time and O(n) space."

"I can do better: Each subclass overrides `speak()`; the caller doesn't care which animal it has. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `names = ['dog', 'cat']`

1. Animal() returns None
2. Dog() returns None
3. Cat() returns None
4. Cow() returns None
5. speak() returns 'Woof'
6. speak() returns 'Meow'
7. sounds() returns ['Woof', 'Meow']

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/oop/animal-sounds)

<a id="q8"></a>
### Q8. Equal Records

🟢 Warm-up · *@dataclass equality*

Return True if two (name, id) records describe the same person, comparing objects with ==.

**Examples**

- `a = ('Ana', 1), b = ('Ana', 1)` → `True`
- `a = ('Ana', 1), b = ('Ana', 2)` → `False`

In [ ]:
def same_person(a, b):
    """
    Return True if two (name, id) records describe the same person, comparing
    objects with ==.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(same_person, [
    ((('Ana', 1), ('Ana', 1)), True),
    ((('Ana', 1), ('Ana', 2)), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A plain class: == compares identity, so define `__eq__` by hand.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`@dataclass` generates `__init__`, `__repr__` and `__eq__` automatically.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** A plain class: == compares identity, so define `__eq__` by hand.

```python
def same_person(a, b):
    class Record:
        def __init__(self, name, ident):
            self.name = name
            self.ident = ident

        def __eq__(self, other):
            return self.name == other.name and self.ident == other.ident

    return Record(*a) == Record(*b)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `@dataclass` generates `__init__`, `__repr__` and `__eq__` automatically.

```python
def same_person(a, b):
    from dataclasses import dataclass

    @dataclass
    class Record:
        name: str
        ident: int

    return Record(*a) == Record(*b)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A plain class: == compares identity, so define `__eq__` by hand. That costs O(1) time and O(1) space."

"I can do better: `@dataclass` generates `__init__`, `__repr__` and `__eq__` automatically. That's O(1) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `a = ('Ana', 1), b = ('Ana', 1)`

1. Record() returns None
2. \_\_annotate\_\_() returns {'name': &lt;class 'str'&gt;, 'ident': &lt;cla...
3. same\_person() returns True

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/oop/equal-records)

<a id="q9"></a>
### Q9. Temperature With a Property

🟡 Interview standard · *@property*

`Temperature(celsius)`. Reading `fahrenheit` converts; setting `celsius` updates it. Use methods `get_f()` and `set_c(c)` in the tests.

**Examples**

- `Temperature(100) → get_f() → set_c(0) → get_f()` → `[212.0, None, 32.0]`

In [ ]:
class Temperature:
    """
    Temperature(celsius). Reading fahrenheit converts; setting celsius updates
    it. Use methods get_f() and set_c(c) in the tests.
    """

    def __init__(self, celsius):
        pass

    def get_f(self):
        pass

    def set_c(self, c):
        pass


# ▶ Run this cell to test your answer
check(Temperature, [
    (((100,), [['get_f', ()], ['set_c', (0,)], ['get_f', ()]]), [212.0, None, 32.0]),
], design=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Store both values and keep them in sync by hand in every setter.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Store only Celsius and compute Fahrenheit with a `@property` — the two can never disagree.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Store both values and keep them in sync by hand in every setter.

```python
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius
        self.fahrenheit = celsius * 9 / 5 + 32

    def set_c(self, c):
        self.celsius = c
        self.fahrenheit = c * 9 / 5 + 32

    def get_f(self):
        return self.fahrenheit
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Store only Celsius and compute Fahrenheit with a `@property` — the two can never disagree.

```python
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def fahrenheit(self):
        return self.celsius * 9 / 5 + 32

    def set_c(self, c):
        self.celsius = c

    def get_f(self):
        return self.fahrenheit
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Store both values and keep them in sync by hand in every setter. That costs O(1) time and O(1) space."

"I can do better: Store only Celsius and compute Fahrenheit with a `@property` — the two can never disagree. That's O(1) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `Temperature(100) → get_f() → set_c(0) → get_f()`

1. \_\_init\_\_() returns None
2. fahrenheit() returns 212.0
3. get\_f() returns 212.0
4. set\_c() returns None
5. fahrenheit() returns 32.0
6. get\_f() returns 32.0

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/oop/temperature-with-a-property)

<a id="q10"></a>
### Q10. Simple Queue Class

🟡 Interview standard · *deque inside a class*

`Queue()` with `enqueue(x)`, `dequeue()` (None if empty) and `size()`.

**Examples**

- `Queue() → enqueue('a') → enqueue('b') → dequeue() → size() → dequeue() → dequeue()` → `[None, None, 'a', 1, 'b', None]`

In [ ]:
class Queue:
    """
    Queue() with enqueue(x), dequeue() (None if empty) and size().
    """

    def __init__(self):
        pass

    def enqueue(self, x):
        pass

    def dequeue(self):
        pass

    def size(self):
        pass


# ▶ Run this cell to test your answer
check(Queue, [
    (((), [['enqueue', ('a',)], ['enqueue', ('b',)], ['dequeue', ()], ['size', ()], ['dequeue', ()], ['dequeue', ()]]), [None, None, 'a', 1, 'b', None]),
], design=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A list with `pop(0)`, which shifts every item.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`collections.deque` removes from the front in O(1).

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** A list with `pop(0)`, which shifts every item.

```python
class Queue:
    def __init__(self):
        self.items = []

    def enqueue(self, x):
        self.items.append(x)

    def dequeue(self):
        if len(self.items) == 0:
            return None
        return self.items.pop(0)

    def size(self):
        return len(self.items)
```

**Cost:** O(n) dequeue time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `collections.deque` removes from the front in O(1).

```python
class Queue:
    def __init__(self):
        from collections import deque
        self.items = deque()

    def enqueue(self, x):
        self.items.append(x)

    def dequeue(self):
        return self.items.popleft() if self.items else None

    def size(self):
        return len(self.items)
```

**Cost:** O(1) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A list with `pop(0)`, which shifts every item. That costs O(n) dequeue time and O(n) space."

"I can do better: `collections.deque` removes from the front in O(1). That's O(1) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `Queue() → enqueue('a') → enqueue('b') → dequeue() → size() → dequeue() → dequeue()`

1. \_\_init\_\_() returns None
2. enqueue() returns None
3. enqueue() returns None
4. dequeue() returns 'a'
5. size() returns 1
6. dequeue() returns 'b'
7. dequeue() returns None

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/oop/simple-queue-class)

<a id="q11"></a>
### Q11. Shopping Cart

🟡 Interview standard · *Objects holding dicts*

`Cart()` with `add(item, price, qty=1)`, `remove(item)` and `total()` (rounded to 2 decimals). Adding an item again increases its quantity.

**Examples**

- `Cart() → add('pen', 10, 2) → add('ink', 25) → add('pen', 10) → total() → remove('ink') → total()` → `[None, None, None, 55, None, 30]`

In [ ]:
class Cart:
    """
    Cart() with add(item, price, qty=1), remove(item) and total() (rounded to 2
    decimals). Adding an item again increases its quantity.
    """

    def __init__(self):
        pass

    def add(self, item, price, qty=1):
        pass

    def total(self):
        pass

    def remove(self, item):
        pass


# ▶ Run this cell to test your answer
check(Cart, [
    (((), [['add', ('pen', 10, 2)], ['add', ('ink', 25)], ['add', ('pen', 10)], ['total', ()], ['remove', ('ink',)], ['total', ()]]), [None, None, None, 55, None, 30]),
], design=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Store a list of (item, price) entries, one per unit, and scan it for removal and totals.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Store item → [price, qty] in a dict.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Store a list of (item, price) entries, one per unit, and scan it for removal and totals.

```python
class Cart:
    def __init__(self):
        self.units = []

    def add(self, item, price, qty=1):
        for _ in range(qty):
            self.units.append((item, price))

    def remove(self, item):
        self.units = [u for u in self.units if u[0] != item]

    def total(self):
        t = 0
        for name, price in self.units:
            t += price
        return round(t, 2)
```

**Cost:** O(n) per call time, O(units) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Store item → [price, qty] in a dict.

```python
class Cart:
    def __init__(self):
        self.items = {}

    def add(self, item, price, qty=1):
        old_qty = self.items.get(item, [price, 0])[1]
        self.items[item] = [price, old_qty + qty]

    def remove(self, item):
        self.items.pop(item, None)

    def total(self):
        return round(sum(p * q for p, q in self.items.values()), 2)
```

**Cost:** O(1) add/remove, O(k) total time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Store a list of (item, price) entries, one per unit, and scan it for removal and totals. That costs O(n) per call time and O(units) space."

"I can do better: Store item → [price, qty] in a dict. That's O(1) add/remove, O(k) total time and O(k) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `Cart() → add('pen', 10, 2) → add('ink', 25) → add('pen', 10) → total() → remove('ink') → total()`

1. \_\_init\_\_() returns None
2. add() returns None
3. add() returns None
4. add() returns None
5. total() returns 55
6. remove() returns None
7. total() returns 30

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/oop/shopping-cart)

<a id="q12"></a>
### Q12. Library Loans

🟡 Interview standard · *Sets inside objects*

`Library(books)` with `borrow(title)` (True if available, then it's out), `give_back(title)` and `available()` (sorted list).

**Examples**

- `Library(['Dune', 'Emma']) → borrow('Dune') → borrow('Dune') → available() → give_back('Dune') → available()` → `[True, False, ['Emma'], None, ['Dune', 'Emma']]`

In [ ]:
class Library:
    """
    Library(books) with borrow(title) (True if available, then it's out),
    give_back(title) and available() (sorted list).
    """

    def __init__(self, books):
        pass

    def borrow(self, title):
        pass

    def available(self):
        pass

    def give_back(self, title):
        pass


# ▶ Run this cell to test your answer
check(Library, [
    (((['Dune', 'Emma'],), [['borrow', ('Dune',)], ['borrow', ('Dune',)], ['available', ()], ['give_back', ('Dune',)], ['available', ()]]), [True, False, ['Emma'], None, ['Dune', 'Emma']]),
], design=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A list of [title, is\_out] pairs, scanned on every call.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Two sets: the books on the shelf and the books out.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** A list of [title, is\_out] pairs, scanned on every call.

```python
class Library:
    def __init__(self, books):
        self.books = [[b, False] for b in books]

    def borrow(self, title):
        for entry in self.books:
            if entry[0] == title and not entry[1]:
                entry[1] = True
                return True
        return False

    def give_back(self, title):
        for entry in self.books:
            if entry[0] == title:
                entry[1] = False

    def available(self):
        return sorted(e[0] for e in self.books if not e[1])
```

**Cost:** O(n) per call time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Two sets: the books on the shelf and the books out.

```python
class Library:
    def __init__(self, books):
        self.shelf = set(books)
        self.out = set()

    def borrow(self, title):
        if title not in self.shelf:
            return False
        self.shelf.remove(title)
        self.out.add(title)
        return True

    def give_back(self, title):
        if title in self.out:
            self.out.remove(title)
            self.shelf.add(title)

    def available(self):
        return sorted(self.shelf)
```

**Cost:** O(1) borrow/return time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A list of [title, is\_out] pairs, scanned on every call. That costs O(n) per call time and O(n) space."

"I can do better: Two sets: the books on the shelf and the books out. That's O(1) borrow/return time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `Library(['Dune', 'Emma']) → borrow('Dune') → borrow('Dune') → available() → give_back('Dune') → available()`

1. \_\_init\_\_() returns None
2. borrow() returns True
3. borrow() returns False
4. available() returns ['Emma']
5. give\_back() returns None
6. available() returns ['Dune', 'Emma']

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/oop/library-loans)

<a id="q13"></a>
### Q13. Protected Age

🟡 Interview standard · *Encapsulation and validation*

`Person(name, age)` with `set_age(a)` that only accepts 0–150 (returns True/False) and `get_age()`.

**Examples**

- `Person('Ana', 20) → set_age(21) → set_age(-5) → get_age()` → `[True, False, 21]`

In [ ]:
class Person:
    """
    Person(name, age) with set_age(a) that only accepts 0–150 (returns
    True/False) and get_age().
    """

    def __init__(self, name, age):
        pass

    def set_age(self, a):
        pass

    def get_age(self):
        pass


# ▶ Run this cell to test your answer
check(Person, [
    ((('Ana', 20), [['set_age', (21,)], ['set_age', (-5,)], ['get_age', ()]]), [True, False, 21]),
], design=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Public attribute plus a separate validate function called from the setter.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Keep the value in `_age` (internal by convention) and guard every change in one method.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Public attribute plus a separate validate function called from the setter.

```python
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def valid(self, a):
        if a < 0:
            return False
        if a > 150:
            return False
        return True

    def set_age(self, a):
        ok = self.valid(a)
        if ok:
            self.age = a
        return ok

    def get_age(self):
        return self.age
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Keep the value in `_age` (internal by convention) and guard every change in one method.

```python
class Person:
    def __init__(self, name, age):
        self.name = name
        self._age = age

    def set_age(self, a):
        if not 0 <= a <= 150:
            return False
        self._age = a
        return True

    def get_age(self):
        return self._age
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Public attribute plus a separate validate function called from the setter. That costs O(1) time and O(1) space."

"I can do better: Keep the value in `_age` (internal by convention) and guard every change in one method. That's O(1) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `Person('Ana', 20) → set_age(21) → set_age(-5) → get_age()`

1. \_\_init\_\_() returns None
2. set\_age() returns True
3. set\_age() returns False
4. get\_age() returns 21

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/oop/protected-age)

<a id="q14"></a>
### Q14. Playlist With len and in

🟡 Interview standard · *Special methods*

`Playlist()` with `add(song)`, `count()` (number of songs) and `has(song)`. The optimal answer makes `len(p)` and `song in p` work too.

**Examples**

- `Playlist() → add('Hey') → add('Yo') → count() → has('Yo') → has('Ok')` → `[None, None, 2, True, False]`

In [ ]:
class Playlist:
    """
    Playlist() with add(song), count() (number of songs) and has(song). The
    optimal answer makes len(p) and song in p work too.
    """

    def __init__(self):
        pass

    def add(self, song):
        pass

    def count(self):
        pass

    def has(self, song):
        pass


# ▶ Run this cell to test your answer
check(Playlist, [
    (((), [['add', ('Hey',)], ['add', ('Yo',)], ['count', ()], ['has', ('Yo',)], ['has', ('Ok',)]]), [None, None, 2, True, False]),
], design=True)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Ordinary methods only.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Define `__len__` and `__contains__` so the object behaves like a built-in collection.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Ordinary methods only.

```python
class Playlist:
    def __init__(self):
        self.songs = []

    def add(self, song):
        self.songs.append(song)

    def count(self):
        n = 0
        for _ in self.songs:
            n += 1
        return n

    def has(self, song):
        for s in self.songs:
            if s == song:
                return True
        return False
```

**Cost:** O(n) has time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Define `__len__` and `__contains__` so the object behaves like a built-in collection.

```python
class Playlist:
    def __init__(self):
        self.songs = []

    def add(self, song):
        self.songs.append(song)

    def __len__(self):
        return len(self.songs)

    def __contains__(self, song):
        return song in self.songs

    def count(self):
        return len(self)

    def has(self, song):
        return song in self
```

**Cost:** O(n) has time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Ordinary methods only. That costs O(n) has time and O(n) space."

"I can do better: Define `__len__` and `__contains__` so the object behaves like a built-in collection. That's O(n) has time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `Playlist() → add('Hey') → add('Yo') → count() → has('Yo') → has('Ok')`

1. \_\_init\_\_() returns None
2. add() returns None
3. add() returns None
4. \_\_len\_\_() returns 2
5. count() returns 2
6. \_\_contains\_\_() returns True
7. has() returns True
8. \_\_contains\_\_() returns False
9. has() returns False

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/oop/playlist-with-len-and-in)

<a id="q15"></a>
### Q15. Add Two Points

🟡 Interview standard · *Operator overloading*

Given two points as tuples, return their sum as a tuple — using a Point class in both answers.

**Examples**

- `a = (1, 2), b = (3, 4)` → `(4, 6)`
- `a = (0, 0), b = (-1, 5)` → `(-1, 5)`

In [ ]:
def add_points(a, b):
    """
    Given two points as tuples, return their sum as a tuple — using a Point
    class in both answers.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(add_points, [
    (((1, 2), (3, 4)), (4, 6)),
    (((0, 0), (-1, 5)), (-1, 5)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A class with an `add` method you call by name.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Define `__add__` so `p + q` works.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** A class with an `add` method you call by name.

```python
def add_points(a, b):
    class Point:
        def __init__(self, x, y):
            self.x = x
            self.y = y

        def add(self, other):
            return Point(self.x + other.x, self.y + other.y)

    p = Point(*a).add(Point(*b))
    return (p.x, p.y)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Define `__add__` so `p + q` works.

```python
def add_points(a, b):
    class Point:
        def __init__(self, x, y):
            self.x, self.y = x, y

        def __add__(self, other):
            return Point(self.x + other.x, self.y + other.y)

    p = Point(*a) + Point(*b)
    return (p.x, p.y)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A class with an `add` method you call by name. That costs O(1) time and O(1) space."

"I can do better: Define `__add__` so `p + q` works. That's O(1) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `a = (1, 2), b = (3, 4)`

1. Point() returns None
2. \_\_init\_\_() returns None
3. \_\_init\_\_() returns None
4. \_\_init\_\_() returns None
5. \_\_add\_\_() returns &lt;topic\_11\_oop.add\_points.&lt;locals&gt;.Poi...
6. add\_points() returns (4, 6)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/oop/add-two-points)

<a id="q16"></a>
### Q16. Total Area of Shapes

🟡 Interview standard · *Inheritance and polymorphism*

Given shapes like ('circle', r) and ('square', s), return the total area rounded to 2 decimals.

**Examples**

- `shapes = [('circle', 1), ('square', 2)]` → `7.14`
- `shapes = []` → `0`

In [ ]:
def total_area(shapes):
    """
    Given shapes like ('circle', r) and ('square', s), return the total area
    rounded to 2 decimals.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(total_area, [
    (([('circle', 1), ('square', 2)],), 7.14),
    (([],), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

One function with an if/elif on the shape type.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A base class Shape with subclasses that each define `area()`; the loop just calls `area()` on every object.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** One function with an if/elif on the shape type.

```python
def total_area(shapes):
    total = 0
    for kind, size in shapes:
        if kind == "circle":
            total += 3.141592653589793 * size * size
        elif kind == "square":
            total += size * size
    return round(total, 2)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A base class Shape with subclasses that each define `area()`; the loop just calls `area()` on every object.

```python
def total_area(shapes):
    import math

    class Shape:
        def area(self):
            raise NotImplementedError

    class Circle(Shape):
        def __init__(self, r):
            self.r = r

        def area(self):
            return math.pi * self.r ** 2

    class Square(Shape):
        def __init__(self, s):
            self.s = s

        def area(self):
            return self.s ** 2

    kinds = {"circle": Circle, "square": Square}
    objects = [kinds[k](size) for k, size in shapes]
    return round(sum(shape.area() for shape in objects), 2)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: One function with an if/elif on the shape type. That costs O(n) time and O(1) space."

"I can do better: A base class Shape with subclasses that each define `area()`; the loop just calls `area()` on every object. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `shapes = [('circle', 1), ('square', 2)]`

1. Shape() returns None
2. Circle() returns None
3. Square() returns None
4. \_\_init\_\_() returns None
5. \_\_init\_\_() returns None
6. area() returns 3.141592653589793
7. area() returns 4
8. total\_area() returns 7.14

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/oop/total-area-of-shapes)

<a id="q17"></a>
### Q17. Count Instances

🟡 Interview standard · *Class attributes*

Create `n` Robot objects and return how many were created, counted by the class itself.

**Examples**

- `n = 3` → `3`
- `n = 0` → `0`

In [ ]:
def robots_made(n):
    """
    Create n Robot objects and return how many were created, counted by the
    class itself.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(robots_made, [
    ((3,), 3),
    ((0,), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A counter variable outside the class, incremented by hand after each creation.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A class attribute shared by all instances, incremented in `__init__`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** A counter variable outside the class, incremented by hand after each creation.

```python
def robots_made(n):
    class Robot:
        def __init__(self, serial):
            self.serial = serial

    made = 0
    for i in range(n):
        Robot(i)
        made += 1
    return made
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A class attribute shared by all instances, incremented in `__init__`.

```python
def robots_made(n):
    class Robot:
        count = 0

        def __init__(self):
            Robot.count += 1

    for _ in range(n):
        Robot()
    return Robot.count
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A counter variable outside the class, incremented by hand after each creation. That costs O(n) time and O(1) space."

"I can do better: A class attribute shared by all instances, incremented in `__init__`. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `n = 2`

1. Robot() returns None
2. \_\_init\_\_() returns None
3. \_\_init\_\_() returns None
4. robots\_made() returns 2

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/oop/count-instances)

<a id="q18"></a>
### Q18. Sort Employees by Salary

🟡 Interview standard · *\_\_lt\_\_ for sorting*

Given (name, salary) pairs, return the names sorted by salary using Employee objects.

**Examples**

- `pairs = [('ana', 50), ('ben', 30), ('cy', 40)]` → `['ben', 'cy', 'ana']`

In [ ]:
def by_salary(pairs):
    """
    Given (name, salary) pairs, return the names sorted by salary using Employee
    objects.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(by_salary, [
    (([('ana', 50), ('ben', 30), ('cy', 40)],), ['ben', 'cy', 'ana']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Pass a key function that reads the attribute.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Define `__lt__` so the objects themselves know how to compare; then plain `sorted()` works.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Pass a key function that reads the attribute.

```python
def by_salary(pairs):
    class Employee:
        def __init__(self, name, salary):
            self.name = name
            self.salary = salary

    staff = [Employee(n, s) for n, s in pairs]
    staff.sort(key=lambda e: e.salary)
    return [e.name for e in staff]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Define `__lt__` so the objects themselves know how to compare; then plain `sorted()` works.

```python
def by_salary(pairs):
    class Employee:
        def __init__(self, name, salary):
            self.name, self.salary = name, salary

        def __lt__(self, other):
            return self.salary < other.salary

    return [e.name for e in sorted(Employee(n, s) for n, s in pairs)]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Pass a key function that reads the attribute. That costs O(n log n) time and O(n) space."

"I can do better: Define `__lt__` so the objects themselves know how to compare; then plain `sorted()` works. That's O(n log n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `pairs = [('ana', 50), ('ben', 30), ('cy', 40)]`

1. Employee() returns None
2. \_\_init\_\_() returns None
3. \_\_init\_\_() returns None
4. \_\_init\_\_() returns None
5. \_\_lt\_\_() returns True
6. \_\_lt\_\_() returns False
7. \_\_lt\_\_() returns True
8. \_\_lt\_\_() returns True
9. \_\_lt\_\_() returns True
10. \_\_lt\_\_() returns False
11. by\_salary() returns ['ben', 'cy', 'ana']

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/oop/sort-employees-by-salary)

<a id="q19"></a>
### Q19. Date From Text

🟡 Interview standard · *@classmethod constructors*

Parse 'DD-MM-YYYY' into a Date object and return (year, month, day).

**Examples**

- `text = '25-12-2024'` → `(2024, 12, 25)`
- `text = '01-01-2000'` → `(2000, 1, 1)`

In [ ]:
def parse_date(text):
    """
    Parse 'DD-MM-YYYY' into a Date object and return (year, month, day).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(parse_date, [
    (('25-12-2024',), (2024, 12, 25)),
    (('01-01-2000',), (2000, 1, 1)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Parse outside the class, then call the normal constructor.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

An alternative constructor `Date.from_string(text)` as a `@classmethod`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Parse outside the class, then call the normal constructor.

```python
def parse_date(text):
    class Date:
        def __init__(self, day, month, year):
            self.day, self.month, self.year = day, month, year

    parts = text.split("-")
    d = Date(int(parts[0]), int(parts[1]), int(parts[2]))
    return (d.year, d.month, d.day)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** An alternative constructor `Date.from_string(text)` as a `@classmethod`.

```python
def parse_date(text):
    class Date:
        def __init__(self, day, month, year):
            self.day, self.month, self.year = day, month, year

        @classmethod
        def from_string(cls, s):
            day, month, year = map(int, s.split("-"))
            return cls(day, month, year)

    d = Date.from_string(text)
    return (d.year, d.month, d.day)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Parse outside the class, then call the normal constructor. That costs O(1) time and O(1) space."

"I can do better: An alternative constructor `Date.from_string(text)` as a `@classmethod`. That's O(1) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `text = '25-12-2024'`

1. Date() returns None
2. \_\_init\_\_() returns None
3. from\_string() returns &lt;topic\_11\_oop.parse\_date.&lt;locals&gt;.Dat...
4. parse\_date() returns (2024, 12, 25)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/oop/date-from-text)

<a id="q20"></a>
### Q20. Car With an Engine

🟡 Interview standard · *Composition*

A Car has an Engine. Return the messages from starting the car twice: ['Engine started', 'Already running'].

**Examples**

- `` → `['Engine started', 'Already running']`

In [ ]:
def start_twice():
    """
    A Car has an Engine. Return the messages from starting the car twice:
    ['Engine started', 'Already running'].
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(start_twice, [
    ((), ['Engine started', 'Already running']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

One big class that tracks the engine state with its own flags.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Composition: Car holds an Engine object and delegates `start()` to it — each class has one job.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** One big class that tracks the engine state with its own flags.

```python
def start_twice():
    class Car:
        def __init__(self):
            self.engine_on = False

        def start(self):
            if self.engine_on:
                return "Already running"
            self.engine_on = True
            return "Engine started"

    car = Car()
    return [car.start(), car.start()]
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Composition: Car holds an Engine object and delegates `start()` to it — each class has one job.

```python
def start_twice():
    class Engine:
        def __init__(self):
            self.running = False

        def start(self):
            if self.running:
                return "Already running"
            self.running = True
            return "Engine started"

    class Car:
        def __init__(self):
            self.engine = Engine()

        def start(self):
            return self.engine.start()

    car = Car()
    return [car.start(), car.start()]
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: One big class that tracks the engine state with its own flags. That costs O(1) time and O(1) space."

"I can do better: Composition: Car holds an Engine object and delegates `start()` to it — each class has one job. That's O(1) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: ``

1. Engine() returns None
2. Car() returns None
3. \_\_init\_\_() returns None
4. \_\_init\_\_() returns None
5. start() returns 'Engine started'
6. start() returns 'Engine started'
7. start() returns 'Already running'
8. start() returns 'Already running'
9. start\_twice() returns ['Engine started', 'Already running']

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/oop/car-with-an-engine)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. Class vs object?</b></summary>

A class is the blueprint (a type); an object is an instance created from it, with its own attribute values.

</details>

<details>
<summary><b>2. What is self?</b></summary>

The instance on which a method is called. Python passes it automatically: `obj.method(x)` is `Class.method(obj, x)`. The name is a convention.

</details>

<details>
<summary><b>3. What does \_\_init\_\_ do? Is it a constructor?</b></summary>

It initialises a newly created object's attributes. The object itself is created by `__new__` just before; in everyday speech `__init__` is called the constructor.

</details>

<details>
<summary><b>4. Class attribute vs instance attribute?</b></summary>

A class attribute is defined in the class body and shared by all instances. An instance attribute is set on `self` and belongs to one object. Assigning `obj.attr = …` creates an instance attribute that hides the class one.

</details>

<details>
<summary><b>5. @staticmethod vs @classmethod?</b></summary>

A classmethod receives the class as `cls` and can create instances (alternative constructors) or use class state. A staticmethod receives nothing automatically; it's just a function grouped inside the class.

</details>

<details>
<summary><b>6. What are the four pillars of OOP?</b></summary>

Encapsulation, abstraction, inheritance and polymorphism.

</details>

<details>
<summary><b>7. Does Python support private attributes?</b></summary>

Not strictly. `_name` signals 'internal'. `__name` triggers name mangling to `_Class__name`, which avoids accidental clashes in subclasses but can still be accessed.

</details>

<details>
<summary><b>8. What is method overriding? Does Python support overloading?</b></summary>

Overriding: a subclass redefines a parent method. Python does not support overloading by parameter types/counts — the last definition wins — so use default arguments or \*args.

</details>

<details>
<summary><b>9. What is MRO?</b></summary>

Method Resolution Order: the order Python searches classes for an attribute, especially with multiple inheritance. It's computed with C3 linearisation and visible as `Class.__mro__`.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Classes and objects
- [ ] I can explain: Methods that change an object's state
- [ ] I can explain: Class attributes, @classmethod and @staticmethod
- [ ] I can explain: Encapsulation and properties
- [ ] I can explain: Inheritance and polymorphism
- [ ] I can explain: Special (dunder) methods

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Total Area of Shapes
- [ ] [Q17](#q17) Count Instances
- [ ] [Q18](#q18) Sort Employees by Salary
- [ ] [Q19](#q19) Date From Text
- [ ] [Q20](#q20) Car With an Engine

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.